In [12]:
import pandas as pd
import os
from torch_measure.data import ResponseMatrix
from torch_measure_ext.beta_twopl import XBetaTwoPL

bbh_items = [
    'boolean_expressions', 'causal_judgement', 'date_understanding', 'disambiguation_qa',
    'formal_fallacies', 'geometric_shapes', 'hyperbaton', 'logical_deduction_5obj',
    'logical_deduction_7obj', 'logical_deduction_3obj', 'movie_recommendation',
    'navigate', 'object_counting', 'penguins_in_table', 'reasoning_colored_obj',
    'ruin_names', 'salient_translation', 'snarks', 'sports_understanding',
    'temporal_sequences', 'tracking_5obj', 'tracking_7obj', 'tracking_3obj', 'web_of_lies'
]

math_items = ['algebra', 'counting_and_prob', 'geometry', 'intermediate_algebra', 'num_theory', 'prealgebra',
              'precalculus']

musr_items = ['murder_mysteries', 'object_placements', 'team_allocation']

gpqa_items = ['diamond', 'extended', 'main']


BENCH = "bbh"
items = bbh_items


def read_responses():
    df = pd.read_parquet("../data_raw_clean/models.parquet")[["fullname", "Architecture", "#Params (B)", "cohort"]]
    df = df.drop_duplicates(subset="fullname", keep="first")
    df = df[df["Architecture"] == ARCH]
    df = df[df["cohort"] == COHORT]

    scores = pd.read_parquet(f"../data_raw_clean/{BENCH}_scores_unique.parquet")

    merged = df.merge(scores, left_on="fullname", right_on="model_name", how="inner").drop(
        columns=['fullname', 'evaluation_date', 'evaluation_date_ts', "Architecture", "#Params (B)", "cohort"])

    score_df = merged.set_index('model_name')
    rm = ResponseMatrix.from_dataframe(score_df)

    return rm


def save(tensor, name):
    nparr = tensor.detach().numpy()

    path = f'../data_irt/{BENCH}/{BENCH}_{name}.csv'

    if os.path.exists(path):
        df = pd.read_csv(path)
        mask = (df['cohort'] == COHORT) & (df['arch'] == ARCH)
        if mask.any():
            # Update existing row
            df.loc[mask, items] = nparr
            print(f"{name}: Updated existing row where cohort='{COHORT}' and arch='{ARCH}'")
        else:
            # Create new row and append
            new_row = pd.DataFrame([nparr], columns=items)
            new_row['cohort'] = COHORT
            new_row['arch'] = ARCH
            df = pd.concat([df, new_row], ignore_index=True)
            print(f"{name}: Added new row since cohort='{COHORT}' and arch='{ARCH} didn't exist")
    else:
        df = pd.DataFrame([nparr], columns=items)
        df['cohort'] = COHORT
        df['arch'] = ARCH

    df.to_csv(path, index=False)


def pipe():
    # data
    rm = read_responses()

    device = 'cpu'
    data = rm.data.to(device)
    mask = rm.observed_mask.to(device)

    # fit
    twopl = XBetaTwoPL(n_subjects=rm.n_subjects, n_items=rm.n_items, device=device)
    twopl_history = twopl.fit(data, mask=mask, max_epochs=10000, verbose=True)

    # 2PL parameters
    abilities_2pl = twopl.ability.detach().cpu()
    difficulties_2pl = twopl.difficulty.detach().cpu()
    discriminations_2pl = twopl.discrimination.detach().cpu()

    print(f"2PL abilities:         min = {abilities_2pl.min():.2f}, max = {abilities_2pl.max():.2f}")
    print(f"2PL difficulties:      min = {difficulties_2pl.min():.2f}, max = {difficulties_2pl.max():.2f}")
    print(f"2PL discriminations:   min = {discriminations_2pl.min():.2f}, max = {discriminations_2pl.max():.2f}, "
          f"mean = {discriminations_2pl.mean():.2f}")

    # save
    save(discriminations_2pl, "discriminations_2pl")
    save(difficulties_2pl, "difficulties_2pl")

In [11]:
for a in [
    'LlamaForCausalLM',
    'Qwen2ForCausalLM',
    'MistralForCausalLM',
    'Gemma2ForCausalLM'
]:
    for c in ['Q1-2025', 'Q4-2024', 'Q3-2024', 'Q2-2024']:
        COHORT = c
        ARCH = a

        pipe()

MLE fitting:  17%|█▋        | 1660/10000 [00:01<00:08, 955.94it/s, loss=-0.989832]


2PL abilities:         min = -1.08, max = 0.50
2PL difficulties:      min = 0.62, max = 0.93
2PL discriminations:   min = 0.53, max = 0.67, mean = 0.58


MLE fitting:   6%|▌         | 603/10000 [00:00<00:09, 1023.11it/s, loss=-0.999683]


2PL abilities:         min = -1.00, max = 0.38
2PL difficulties:      min = 0.51, max = 1.49
2PL discriminations:   min = 0.41, max = 0.82, mean = 0.58
discriminations_2pl: Added new row since cohort='Q4-2024' and arch='LlamaForCausalLM didn't exist
difficulties_2pl: Added new row since cohort='Q4-2024' and arch='LlamaForCausalLM didn't exist


MLE fitting:   6%|▌         | 596/10000 [00:00<00:07, 1332.11it/s, loss=-1.004613]


...

2PL abilities:         min = -0.22, max = 0.32
2PL difficulties:      min = 0.70, max = 0.86
2PL discriminations:   min = 0.89, max = 1.11, mean = 0.97
discriminations_2pl: Added new row since cohort='Q4-2024' and arch='Gemma2ForCausalLM didn't exist
difficulties_2pl: Added new row since cohort='Q4-2024' and arch='Gemma2ForCausalLM didn't exist


MLE fitting:  10%|█         | 1011/10000 [00:00<00:06, 1457.51it/s, loss=-0.976278]


2PL abilities:         min = -0.86, max = -0.03
2PL difficulties:      min = 0.68, max = 1.34
2PL discriminations:   min = 0.41, max = 0.67, mean = 0.52
discriminations_2pl: Added new row since cohort='Q3-2024' and arch='Gemma2ForCausalLM didn't exist
difficulties_2pl: Added new row since cohort='Q3-2024' and arch='Gemma2ForCausalLM didn't exist


MLE fitting:   5%|▍         | 478/10000 [00:00<00:06, 1420.13it/s, loss=-0.948889]


2PL abilities:         min = -0.44, max = -0.23
2PL difficulties:      min = 0.22, max = 0.66
2PL discriminations:   min = 0.54, max = 0.94, mean = 0.74
discriminations_2pl: Added new row since cohort='Q2-2024' and arch='Gemma2ForCausalLM didn't exist
difficulties_2pl: Added new row since cohort='Q2-2024' and arch='Gemma2ForCausalLM didn't exist
